In [1]:
# ============ 第一步：先设置环境变量（必须在导入任何库之前）============
import os

# 设置环境变量（必须在最前面）
os.environ['IR_DATASETS_HOME'] = '../../Trec-llm/dataset'
os.environ['HF_HOME'] = '../../Trec-llm/huggingface_cache'

print(f"IR_DATASETS_HOME 设置为: {os.environ['IR_DATASETS_HOME']}")
print(f"HF_HOME 设置为: {os.environ['HF_HOME']}")

# ============ 第二步：然后才导入其他库 ============
import random
import numpy as np
import torch
import pandas as pd
import pyterrier as pt
from pyterrier.measures import *
from collections import defaultdict
import ir_datasets
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

IR_DATASETS_HOME 设置为: ../../Trec-llm/dataset
HF_HOME 设置为: ../../Trec-llm/huggingface_cache


In [2]:
dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-2019/judged')
df_colbert = pt.io.read_results("../ColBERT-PRF-VirtualAppendix/ColBERT E2E/E2E.2019.res.gz")
evalMeasuresDict = pt.Evaluate(
  df_colbert,
  dataset.get_qrels(), 
  metrics=[ nDCG@1, nDCG@10, nDCG@50, nDCG@100 ]
)
print(evalMeasuresDict)

{'nDCG@1': 0.7325581395348837, 'nDCG@10': 0.6934074729307355, 'nDCG@50': 0.6141665697446085, 'nDCG@100': 0.6029510284363208}


In [3]:
df_colbert

,qid,docno,rank,score,name
0,1037798,1597,3073,14.319312,pyterrier
1,1037798,7141,2542,14.801564,pyterrier
2,1037798,7947,3052,14.339487,pyterrier
3,1037798,9661,2094,15.231604,pyterrier
4,1037798,11366,1288,16.182770,pyterrier
...,...,...,...,...,...
310067,962179,8816879,2659,17.211912,pyterrier
310068,962179,8826601,3768,16.184523,pyterrier
310069,962179,8831025,3280,16.673035,pyterrier
310070,962179,8834789,3706,16.258209,pyterrier


In [3]:
dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-2019/judged')
df_colbert_prf = pt.io.read_results("../ColBERT-PRF-VirtualAppendix/ColBERT-PRF/MSMARCO Passage Res/ColBERT PRF Ranker_beta=1/prf_rank_beta1.2019.res.gz")
evalMeasuresDict = pt.Evaluate(
  df_colbert_prf,
  dataset.get_qrels(), 
  metrics=[ nDCG@1, nDCG@10, nDCG@50, nDCG@100 ]
)
print(evalMeasuresDict)

{'nDCG@1': 0.7868217054263567, 'nDCG@10': 0.7351525358046614, 'nDCG@50': 0.6890168638568059, 'nDCG@100': 0.6902757382204006}


In [10]:
from typing import List
import pandas as pd


def get_easy_negatives(
    dataset, 
    qid: str, 
    df_A: pd.DataFrame,
    df_B: pd.DataFrame,
    num_negatives: int = 100,
    strategy: str = 'min'
) -> List[str]:
    """
    从两个系统中提取真正的"简单"负样本（两个系统都认为不相关）
    
    Args:
        dataset: 数据集对象，需要有get_qrels()方法
        qid: 查询ID
        df_A: 系统A的结果DataFrame (包含qid, docno, score列)
        df_B: 系统B的结果DataFrame (包含qid, docno, score列)
        num_negatives: 需要返回的负样本数量
        strategy: 选择策略
            - 'min': 选择两个系统中score较小者（更保守，确保两边都不相关）
            - 'max': 选择两个系统中score较大者
            - 'mean': 使用平均分
    
    Returns:
        docno列表
    """
    qrels = dataset.get_qrels()
    
    # 筛选当前qid且label=0的文档
    neg_docs = qrels[(qrels['qid'] == qid) & (qrels['label'] == 0)].copy()
    
    if len(neg_docs) == 0:
        print(f"Warning: qid={qid} 没有找到任何负样本 (label=0)")
        return []
    
    # 获取当前qid的两个系统的结果
    df_A_qid = df_A[df_A['qid'] == qid].set_index('docno')
    df_B_qid = df_B[df_B['qid'] == qid].set_index('docno')
    
    # 为每个负样本计算综合score
    scores = []
    for docno in neg_docs['docno']:
        score_A = df_A_qid.loc[docno, 'score'] if docno in df_A_qid.index else 0
        score_B = df_B_qid.loc[docno, 'score'] if docno in df_B_qid.index else 0
        
        if strategy == 'min':
            combined_score = min(score_A, score_B)
        elif strategy == 'max':
            combined_score = max(score_A, score_B)
        elif strategy == 'mean':
            combined_score = (score_A + score_B) / 2
        else:
            combined_score = min(score_A, score_B)  # 默认使用min
        
        scores.append(combined_score)
    
    neg_docs['combined_score'] = scores
    
    # 按综合score升序排序（最低的在前）
    neg_docs_sorted = neg_docs.sort_values('combined_score', ascending=True)
    
    # 返回前num_negatives个docno
    return neg_docs_sorted['docno'].head(num_negatives).tolist()


def rank_based_interleaving(
    df_A: pd.DataFrame, 
    df_B: pd.DataFrame, 
    easy_negatives: dict or list,
    scan_depth: int = 20,
    balance_threshold: int = 2,
    target_docs: int = 9,
    smart_easy_negative: bool = True
) -> pd.DataFrame:
    """
    基于排名优先的交错扫描算法，为每个查询生成包含指定数量文档的最终列表
    
    核心特性：
    1. 平衡性优先：确保A-Only和B-Only文档数量平衡
    2. 智能Easy-Negative：仅在需要时添加，用于"补漆"以实现完美平衡
    
    工作原理：
    - 当Both数量为偶数时，剩余位置是奇数，无法平分 → 添加1个Easy-Negative
    - 当Both数量为奇数时，剩余位置是偶数，可以平分 → 不需要Easy-Negative
    
    Args:
        df_A: 排名A的结果DataFrame (包含qid, docno, score等列)
        df_B: 排名B的结果DataFrame (包含qid, docno, score等列)
        easy_negatives: 简单负样本。可以是:
                       - dict: {qid: [docno_list]} 每个qid对应一个负样本列表
                       - list: [docno_list] 所有qid共享的负样本列表
        scan_depth: 扫描深度，防止无限循环
        balance_threshold: 平衡阈值。当A-Only和B-Only的数量差超过此值时，
                          跳过数量较多的系统的文档，以保持平衡
        target_docs: 最终需要的总文档数量（默认9）
        smart_easy_negative: 是否启用智能Easy-Negative逻辑（默认True）
                            True: 根据Both的奇偶性智能决定是否添加
                            False: 总是尝试填充至target_docs
    
    Returns:
        pd.DataFrame: 包含qid, docno, origin_label, rank列的结果
    """
    # 获取所有唯一的qid
    all_qids = set(df_A['qid'].unique()) | set(df_B['qid'].unique())
    
    # 存储最终结果
    final_results = []
    
    # 逐个查询处理
    for qid in sorted(all_qids):
        # 1. 过滤和排序：获取当前qid的结果
        list_A = df_A[df_A['qid'] == qid].copy()
        list_B = df_B[df_B['qid'] == qid].copy()
        
        # 按score降序排序
        list_A = list_A.sort_values('score', ascending=False).reset_index(drop=True)
        list_B = list_B.sort_values('score', ascending=False).reset_index(drop=True)
        
        # 2. 初始化
        query_selection = []  # 存储选中的文档及其标签
        seen_docs = set()  # 跟踪已添加的docno
        k = 0  # 同步排名指针
        
        # 统计A-Only和B-Only的数量，用于平衡检查
        count_a_only = 0
        count_b_only = 0
        count_both = 0
        
        # 3. 执行交错扫描循环 - 先不考虑Easy-Negative，只收集Both/A-Only/B-Only
        while len(query_selection) < target_docs and k < scan_depth:
            # 检查是否还有文档可以扫描
            if k >= len(list_A) and k >= len(list_B):
                break
            
            # 获取k位置的文档
            doc_A = list_A.iloc[k]['docno'] if k < len(list_A) else None
            doc_B = list_B.iloc[k]['docno'] if k < len(list_B) else None
            
            # 情况1: 两个列表在位置k都有相同的文档 (Both)
            if doc_A is not None and doc_B is not None and doc_A == doc_B:
                if doc_A not in seen_docs:
                    query_selection.append({
                        'qid': qid,
                        'docno': doc_A,
                        'origin_label': 'Both'
                    })
                    seen_docs.add(doc_A)
                    count_both += 1
            
            # 情况2: 两个列表在位置k有不同的文档
            else:
                # 处理A的文档
                if doc_A is not None and doc_A not in seen_docs:
                    # 检查添加A是否会破坏平衡
                    # 只有当A已经比B多太多时才跳过
                    balance_diff = count_a_only - count_b_only
                    
                    if balance_diff >= balance_threshold:
                        # A已经比B多太多，跳过A
                        pass
                    else:
                        query_selection.append({
                            'qid': qid,
                            'docno': doc_A,
                            'origin_label': 'A-Only'
                        })
                        seen_docs.add(doc_A)
                        count_a_only += 1
                        
                        # 检查是否已达到目标（暂时允许达到target_docs）
                        if len(query_selection) == target_docs:
                            break
                
                # 处理B的文档
                if doc_B is not None and doc_B not in seen_docs:
                    # 检查添加B是否会破坏平衡
                    # 只有当B已经比A多太多时才跳过
                    balance_diff = count_b_only - count_a_only
                    
                    if balance_diff >= balance_threshold:
                        # B已经比A多太多，跳过B
                        pass
                    else:
                        query_selection.append({
                            'qid': qid,
                            'docno': doc_B,
                            'origin_label': 'B-Only'
                        })
                        seen_docs.add(doc_B)
                        count_b_only += 1
            
            # 移动指针
            k += 1
        
        # 4. 智能Easy-Negative填充逻辑
        if smart_easy_negative:
            # 核心逻辑：根据Both的奇偶性决定是否需要Easy-Negative
            remaining_slots = target_docs - count_both
            
            if remaining_slots % 2 == 1:
                # 剩余位置是奇数，无法平分给A和B
                # 需要1个Easy-Negative来"补漆"
                needed_easy_negatives = 1
                
                # 如果已经收集了target_docs个文档，需要移除1个来为Easy-Negative腾出空间
                if len(query_selection) == target_docs:
                    # 优先移除数量较多的那一方（A-Only或B-Only）
                    if count_a_only > count_b_only:
                        # 移除最后一个A-Only
                        for i in range(len(query_selection) - 1, -1, -1):
                            if query_selection[i]['origin_label'] == 'A-Only':
                                removed_doc = query_selection.pop(i)
                                seen_docs.remove(removed_doc['docno'])
                                count_a_only -= 1
                                break
                    else:
                        # 移除最后一个B-Only
                        for i in range(len(query_selection) - 1, -1, -1):
                            if query_selection[i]['origin_label'] == 'B-Only':
                                removed_doc = query_selection.pop(i)
                                seen_docs.remove(removed_doc['docno'])
                                count_b_only -= 1
                                break
            else:
                # 剩余位置是偶数，可以平分给A和B
                # 不需要Easy-Negative
                needed_easy_negatives = 0
        else:
            # 非智能模式：总是尝试填充至target_docs
            needed_easy_negatives = target_docs - len(query_selection)
        
        # 5. 添加Easy-Negative文档
        if needed_easy_negatives > 0:
            # 获取当前qid的easy_negatives
            if isinstance(easy_negatives, dict):
                neg_list = easy_negatives.get(qid, [])
            else:
                neg_list = easy_negatives
            
            # 从easy_negatives中选取未见过的文档
            added_negatives = 0
            for neg_doc in neg_list:
                if neg_doc not in seen_docs:
                    query_selection.append({
                        'qid': qid,
                        'docno': neg_doc,
                        'origin_label': 'Easy-Negative'
                    })
                    seen_docs.add(neg_doc)
                    added_negatives += 1
                    
                    if added_negatives >= needed_easy_negatives:
                        break
            
            # 如果easy_negatives不够，发出警告
            if added_negatives < needed_easy_negatives:
                print(f"Warning: qid={qid} 只填充了 {added_negatives} 个Easy-Negative，"
                      f"还缺少 {needed_easy_negatives - added_negatives} 个")
        
        # 6. 添加最终排名
        for rank_idx, item in enumerate(query_selection):
            item['rank'] = rank_idx
            final_results.append(item)
    
    # 转换为DataFrame
    result_df = pd.DataFrame(final_results)
    
    # 确保列顺序
    result_df = result_df[['qid', 'docno', 'rank', 'origin_label']]
    
    return result_df


def analyze_interleaving_results(result_df: pd.DataFrame) -> pd.DataFrame:
    """
    分析交错结果，统计各类别的分布
    
    Args:
        result_df: rank_based_interleaving函数返回的结果
    
    Returns:
        统计信息的DataFrame
    """
    stats = []
    
    for qid in result_df['qid'].unique():
        qid_data = result_df[result_df['qid'] == qid]
        
        stats.append({
            'qid': qid,
            'total_docs': len(qid_data),
            'A-Only': (qid_data['origin_label'] == 'A-Only').sum(),
            'B-Only': (qid_data['origin_label'] == 'B-Only').sum(),
            'Both': (qid_data['origin_label'] == 'Both').sum(),
            'Easy-Negative': (qid_data['origin_label'] == 'Easy-Negative').sum()
        })
    
    stats_df = pd.DataFrame(stats)
    
    # 添加总计行
    total_row = {
        'qid': 'TOTAL',
        'total_docs': stats_df['total_docs'].sum(),
        'A-Only': stats_df['A-Only'].sum(),
        'B-Only': stats_df['B-Only'].sum(),
        'Both': stats_df['Both'].sum(),
        'Easy-Negative': stats_df['Easy-Negative'].sum()
    }
    
    stats_df = pd.concat([stats_df, pd.DataFrame([total_row])], ignore_index=True)
    
    return stats_df


def rank_based_interleaving_with_fairness_check(
    df_A: pd.DataFrame,
    df_B: pd.DataFrame,
    easy_negatives: dict or list,
    scan_depth: int = 20,
    balance_threshold: int = 2,
    target_docs: int = 9,
    smart_easy_negative: bool = True,
    max_imbalance: int = 1
) -> pd.DataFrame:
    """
    带公平性检查的交错算法（推荐用于生产环境）
    
    工作流程：
    1. 先用正常策略处理所有查询
    2. 检测哪些查询不够平衡（A-Only和B-Only差值>max_imbalance）
    3. 对不平衡的查询使用备用策略（轮流选择）重新处理
    4. 返回所有平衡的结果
    
    Args:
        df_A, df_B, easy_negatives, scan_depth, balance_threshold, target_docs, smart_easy_negative:
            与rank_based_interleaving相同
        max_imbalance: 允许的最大不平衡度（默认1，即A和B最多相差1个）
    
    Returns:
        pd.DataFrame: 处理后的结果，所有查询都满足平衡要求
    
    Example:
        >>> result = rank_based_interleaving_with_fairness_check(
        ...     df_colbert, df_colbert_prf, easy_negatives,
        ...     max_imbalance=1  # 要求完全平衡或最多相差1个
        ... )
        ⚠️  检测到不平衡查询: 1121402 (A-Only=2, B-Only=4, 差值=2)
        🔄 使用交替策略重新处理 1 个不平衡查询...
          ✓ 1121402: A-Only=3, B-Only=3, 差值=0
    """
    # 第一次尝试：正常策略
    result = rank_based_interleaving(
        df_A, df_B, easy_negatives,
        scan_depth=scan_depth,
        balance_threshold=balance_threshold,
        target_docs=target_docs,
        smart_easy_negative=smart_easy_negative
    )
    
    # 检查每个查询的平衡度
    stats = analyze_interleaving_results(result)
    
    imbalanced_qids = []
    for _, row in stats.iterrows():
        if row['qid'] == 'TOTAL':
            continue
        
        diff = abs(row['A-Only'] - row['B-Only'])
        if diff > max_imbalance:
            imbalanced_qids.append(row['qid'])
            print(f"⚠️  检测到不平衡查询: {row['qid']} "
                  f"(A-Only={row['A-Only']}, B-Only={row['B-Only']}, 差值={diff})")
    
    if not imbalanced_qids:
        print("✓ 所有查询都已平衡")
        return result
    
    # 对不平衡的查询使用备用策略
    print(f"🔄 使用交替策略重新处理 {len(imbalanced_qids)} 个不平衡查询...")
    
    # 保留平衡的查询
    balanced_result = result[~result['qid'].isin(imbalanced_qids)].copy()
    
    # 重新处理不平衡的查询
    for qid in imbalanced_qids:
        df_A_query = df_A[df_A['qid'] == qid]
        df_B_query = df_B[df_B['qid'] == qid]
        
        # 使用交替策略
        fixed_result = _alternate_strategy(
            df_A_query, df_B_query, qid,
            easy_negatives,
            target_docs=target_docs,
            smart_easy_negative=smart_easy_negative
        )
        
        # 合并结果
        balanced_result = pd.concat([balanced_result, fixed_result], ignore_index=True)
        
        # 显示修正结果
        fixed_stats = analyze_interleaving_results(fixed_result)
        row = fixed_stats[fixed_stats['qid'] == qid].iloc[0]
        print(f"  ✓ {qid}: A-Only={row['A-Only']}, B-Only={row['B-Only']}, "
              f"差值={abs(row['A-Only']-row['B-Only'])}")
    
    return balanced_result.sort_values(['qid', 'rank']).reset_index(drop=True)


def _alternate_strategy(
    df_A: pd.DataFrame,
    df_B: pd.DataFrame,
    qid: str,
    easy_negatives: dict or list,
    target_docs: int = 9,
    smart_easy_negative: bool = True
) -> pd.DataFrame:
    """
    交替选择策略：确保A和B轮流选择文档
    
    适用场景：
    - A和B的排名差异很大
    - 某些文档在一个系统排名很高，在另一个系统排名很低
    - 正常的同步扫描会导致不平衡
    
    工作原理：
    1. 先收集Both文档
    2. 然后A和B轮流选择各自的独特文档
    3. 这样保证A和B的数量完全相等（或最多相差1个）
    """
    # 按score排序
    df_A_sorted = df_A.sort_values('score', ascending=False).reset_index(drop=True)
    df_B_sorted = df_B.sort_values('score', ascending=False).reset_index(drop=True)
    
    # 初始化
    query_selection = []
    seen_docs = set()
    count_both = 0
    count_a_only = 0
    count_b_only = 0
    
    # 先收集Both文档
    k = 0
    max_k = min(len(df_A_sorted), len(df_B_sorted))
    
    while k < max_k and len(query_selection) < target_docs:
        doc_A = df_A_sorted.iloc[k]['docno']
        doc_B = df_B_sorted.iloc[k]['docno']
        
        if doc_A == doc_B and doc_A not in seen_docs:
            query_selection.append({
                'qid': qid,
                'docno': doc_A,
                'origin_label': 'Both'
            })
            seen_docs.add(doc_A)
            count_both += 1
        
        k += 1
    
    # 现在交替选择A和B的独特文档
    turn = 'A'  # 从A开始
    idx_A = 0
    idx_B = 0
    
    while len(query_selection) < target_docs:
        if turn == 'A':
            # A的回合
            added = False
            while idx_A < len(df_A_sorted) and not added:
                doc = df_A_sorted.iloc[idx_A]['docno']
                idx_A += 1
                
                if doc not in seen_docs:
                    query_selection.append({
                        'qid': qid,
                        'docno': doc,
                        'origin_label': 'A-Only'
                    })
                    seen_docs.add(doc)
                    count_a_only += 1
                    added = True
            
            if not added:  # A没有更多文档了，切换到B
                turn = 'B'
                continue
            
            turn = 'B'  # 下一轮是B
        
        else:
            # B的回合
            added = False
            while idx_B < len(df_B_sorted) and not added:
                doc = df_B_sorted.iloc[idx_B]['docno']
                idx_B += 1
                
                if doc not in seen_docs:
                    query_selection.append({
                        'qid': qid,
                        'docno': doc,
                        'origin_label': 'B-Only'
                    })
                    seen_docs.add(doc)
                    count_b_only += 1
                    added = True
            
            if not added:  # B没有更多文档了，切换到A
                turn = 'A'
                continue
            
            turn = 'A'  # 下一轮是A
    
    # 检查是否需要Easy-Negative补漆
    if smart_easy_negative:
        remaining = target_docs - count_both
        if remaining % 2 == 1:
            # 需要补漆，移除1个来腾出空间
            if count_a_only > count_b_only:
                # 移除最后一个A-Only
                for i in range(len(query_selection) - 1, -1, -1):
                    if query_selection[i]['origin_label'] == 'A-Only':
                        query_selection.pop(i)
                        break
            else:
                # 移除最后一个B-Only
                for i in range(len(query_selection) - 1, -1, -1):
                    if query_selection[i]['origin_label'] == 'B-Only':
                        query_selection.pop(i)
                        break
            
            # 添加Easy-Negative
            if isinstance(easy_negatives, dict):
                neg_list = easy_negatives.get(qid, [])
            else:
                neg_list = easy_negatives
            
            for neg_doc in neg_list:
                if neg_doc not in seen_docs:
                    query_selection.append({
                        'qid': qid,
                        'docno': neg_doc,
                        'origin_label': 'Easy-Negative'
                    })
                    break
    
    # 添加排名
    for rank_idx, item in enumerate(query_selection):
        item['rank'] = rank_idx
    
    return pd.DataFrame(query_selection)

In [9]:
dataset.get_qrels()

,qid,docno,label,iteration
0,19335,1017759,0,Q0
1,19335,1082489,0,Q0
2,19335,109063,0,Q0
3,19335,1160863,0,Q0
4,19335,1160871,0,Q0
...,...,...,...,...
9255,1133167,8839920,2,Q0
9256,1133167,8839922,2,Q0
9257,1133167,944810,0,Q0
9258,1133167,949411,0,Q0


In [11]:
# 1. 为所有qid提取easy negatives（使用两个系统取最小score）
all_qids = set(df_colbert['qid'].unique()) | set(df_colbert_prf['qid'].unique())

easy_negatives = {}
for qid in all_qids:
    easy_negatives[qid] = get_easy_negatives(
        dataset=dataset,
        qid=qid,
        df_A=df_colbert,
        df_B=df_colbert_prf,
        num_negatives=100,
        strategy='min'  # 使用两个系统中score较小者
    )

# 2. 运行交错算法
result = rank_based_interleaving_with_fairness_check(
    df_A=df_colbert,
    df_B=df_colbert_prf,
    easy_negatives=easy_negatives,
    scan_depth=20,
    balance_threshold=2,
    target_docs=9,
    smart_easy_negative=True,
    max_imbalance=1
)

⚠️  检测到不平衡查询: 1114819 (A-Only=2, B-Only=4, 差值=2)
⚠️  检测到不平衡查询: 1121402 (A-Only=2, B-Only=4, 差值=2)
🔄 使用交替策略重新处理 2 个不平衡查询...
  ✓ 1114819: A-Only=3, B-Only=3, 差值=0
  ✓ 1121402: A-Only=2, B-Only=2, 差值=0


In [12]:
print("交错结果：")
print(result)
print("\n统计分析：")
print(analysis)
result.to_csv('interleaving_results.csv', index=False)
analysis.to_csv('analysis_summary.csv', index=False)

交错结果：
         qid    docno  rank   origin_label
0    1037798  8760871     0           Both
1    1037798  7822415     1           Both
2    1037798  8760866     2           Both
3    1037798  2360252     3         A-Only
4    1037798  3641634     4         B-Only
..       ...      ...   ...            ...
382   962179  2978869     4         A-Only
383   962179  2329701     5         B-Only
384   962179  4606386     6         A-Only
385   962179  2978871     7         B-Only
386   962179  1006865     8  Easy-Negative

[387 rows x 4 columns]

统计分析：
        qid  total_docs  A-Only  B-Only  Both  Easy-Negative
0   1037798           9       3       3     3              0
1    104861           9       4       4     0              1
2   1063750           9       4       4     1              0
3   1103812           9       4       4     0              1
4   1106007           9       3       3     2              1
5   1110199           9       3       3     3              0
6   1112341         